# GBFS Snapshot Ingestion Job

This notebook ingests **real-time GBFS snapshots** from the Bike Share Toronto public feed and stores them in **managed Unity Catalog bronze tables**.

Its main objective is to create a reliable ingestion layer that captures the current operational state of the bike-sharing system, including:

- system metadata
- station information
- station availability/status
- vehicle types
- discovery feed structure

The notebook also stores the raw JSON files for traceability and refreshes “latest” views used by downstream transformation and prediction jobs.

This notebook represents the **bronze ingestion layer** of the forecasting pipeline.

## Process Overview

This ingestion job performs the following steps:

### 1. Discover available GBFS feeds
The notebook starts from the GBFS auto-discovery endpoint and identifies all published feed URLs.

### 2. Download raw JSON payloads
Each available feed is downloaded and stored as a raw JSON file in DBFS for auditability and reproducibility.

### 3. Normalize and flatten feed records
The JSON payloads are transformed into tabular format so they can be stored as structured bronze tables.

### 4. Add ingestion metadata
Each ingested record is enriched with:
- ingestion timestamp
- UTC epoch
- source URL
- ingestion date/time breakdown

This makes historical snapshot tracking possible.

### 5. Write to Unity Catalog bronze tables
The normalized data is appended into managed Delta tables such as:
- `bronze_gbfs_station_information`
- `bronze_gbfs_station_status`
- `bronze_gbfs_system_information`

### 6. Refresh “latest” views
The notebook creates or refreshes views that expose the most recent record for:
- station status
- station information
- system information

These views are later used by downstream transformation jobs.

In [0]:
# Databricks notebook source
# ============================================================
# GBFS SNAPSHOT INGESTION JOB - SERVERLESS + UNITY CATALOG SAFE
# Bike Share Toronto - every X minutes
#
# Key fixes:
# 1) No spark.sparkContext
# 2) No CREATE TABLE ... LOCATION 'dbfs:/...'
# 3) Writes directly to managed Unity Catalog tables
# ============================================================

from pyspark.sql import functions as F
import requests
import json
import pandas as pd
from datetime import datetime, timezone

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
GBFS_AUTO_DISCOVERY_URL = "https://tor.publicbikesystem.net/customer/gbfs/v2/gbfs.json"

# >>> CHANGE THESE <<<
UC_CATALOG = spark.catalog.currentCatalog()
UC_SCHEMA  = spark.catalog.currentDatabase()

print("Catalog:", UC_CATALOG)
print("Schema :", UC_SCHEMA)

# Optional raw file storage (files only, not UC tables)
BASE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data"
RAW_DIR = f"{BASE_DIR}/raw/gbfs"

REQUEST_TIMEOUT = 30
REQUEST_HEADERS = {
    "User-Agent": "Databricks-GBFS-Ingestion/1.0"
}

# ------------------------------------------------------------
# 1) UC TABLE NAMES
# ------------------------------------------------------------
TBL_DISCOVERY        = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_discovery"
TBL_SYSTEM_INFO      = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_system_information"
TBL_STATION_INFO     = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_station_information"
TBL_STATION_STATUS   = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_station_status"
TBL_FREE_BIKE_STATUS = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_free_bike_status"
TBL_VEHICLE_TYPES    = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_gbfs_vehicle_types"

VW_STATION_STATUS_LATEST = f"{UC_CATALOG}.{UC_SCHEMA}.vw_gbfs_station_status_latest"
VW_STATION_INFO_LATEST   = f"{UC_CATALOG}.{UC_SCHEMA}.vw_gbfs_station_information_latest"
VW_SYSTEM_INFO_LATEST    = f"{UC_CATALOG}.{UC_SCHEMA}.vw_gbfs_system_information_latest"

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def utc_now():
    return datetime.now(timezone.utc)

def fetch_json(url: str):
    r = requests.get(url, headers=REQUEST_HEADERS, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()

def normalize_feed_name(name: str) -> str:
    return (
        str(name).lower()
        .strip()
        .replace(" ", "_")
        .replace("-", "_")
    )

def add_ingestion_columns(df, ingest_ts_str, ingest_epoch, source_url):
    return (
        df.withColumn("ingested_at_utc", F.to_timestamp(F.lit(ingest_ts_str)))
          .withColumn("ingest_epoch_utc", F.lit(int(ingest_epoch)).cast("long"))
          .withColumn("source_url", F.lit(source_url))
          .withColumn("ingest_date", F.to_date(F.col("ingested_at_utc")))
          .withColumn("ingest_year", F.year("ingested_at_utc"))
          .withColumn("ingest_month", F.month("ingested_at_utc"))
          .withColumn("ingest_day", F.dayofmonth("ingested_at_utc"))
          .withColumn("ingest_hour", F.hour("ingested_at_utc"))
          .withColumn("ingest_minute", F.minute("ingested_at_utc"))
    )

def write_raw_json(payload: dict, feed_name: str, ingest_dt: datetime):
    yyyy = ingest_dt.strftime("%Y")
    mm   = ingest_dt.strftime("%m")
    dd   = ingest_dt.strftime("%d")
    hh   = ingest_dt.strftime("%H")
    mi   = ingest_dt.strftime("%M")
    ts   = ingest_dt.strftime("%Y%m%dT%H%M%SZ")

    out_dir = f"{RAW_DIR}/{feed_name}/year={yyyy}/month={mm}/day={dd}/hour={hh}/minute={mi}"
    out_file = f"{out_dir}/{feed_name}_{ts}.json"

    dbutils.fs.put(out_file, json.dumps(payload, ensure_ascii=False), overwrite=True)
    print(f"RAW saved: {out_file}")

def make_json_safe(obj):
    if isinstance(obj, dict):
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    elif isinstance(obj, list):
        return json.dumps(obj, ensure_ascii=False)
    elif isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    else:
        return str(obj)

def flatten_record(record: dict) -> dict:
    out = {}
    for k, v in record.items():
        key = str(k).strip().replace(" ", "_").replace("-", "_")
        out[key] = make_json_safe(v)
    return out

def create_df_from_records(records):
    if not records:
        return None

    flat_rows = [flatten_record(r) for r in records]

    all_cols = set()
    for r in flat_rows:
        all_cols.update(r.keys())
    all_cols = sorted(all_cols)

    normalized_rows = []
    for r in flat_rows:
        normalized_rows.append({c: r.get(c, None) for c in all_cols})

    pdf = pd.DataFrame(normalized_rows, columns=all_cols)
    pdf = pdf.where(pd.notnull(pdf), None)

    return spark.createDataFrame(pdf)

def discover_feeds(gbfs_json: dict):
    feeds = []
    data = gbfs_json.get("data", {})

    if isinstance(data, dict):
        if "en" in data and isinstance(data["en"], dict) and "feeds" in data["en"]:
            feeds = data["en"]["feeds"]
        elif "feeds" in data:
            feeds = data["feeds"]
        else:
            for _, v in data.items():
                if isinstance(v, dict) and "feeds" in v:
                    feeds = v["feeds"]
                    break

    out = {}
    for f in feeds:
        name = normalize_feed_name(f.get("name", ""))
        url = f.get("url")
        if name and url:
            out[name] = url

    return out

def ensure_schema():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")

def write_uc_table(df, table_name: str):
    (
        df.write
          .format("delta")
          .mode("append")
          .saveAsTable(table_name)
    )

# ------------------------------------------------------------
# 3) START
# ------------------------------------------------------------
ensure_schema()

ingest_dt = utc_now()
ingest_ts = ingest_dt.isoformat()
ingest_epoch = int(ingest_dt.timestamp())

print("========================================")
print("GBFS ingestion started")
print(f"Ingest UTC timestamp: {ingest_ts}")
print(f"UC target schema: {UC_CATALOG}.{UC_SCHEMA}")
print("========================================")

# ------------------------------------------------------------
# 4) DISCOVERY
# ------------------------------------------------------------
gbfs_root = fetch_json(GBFS_AUTO_DISCOVERY_URL)
write_raw_json(gbfs_root, "gbfs_discovery", ingest_dt)

feed_map = discover_feeds(gbfs_root)

print("Discovered feeds:")
for k, v in feed_map.items():
    print(f" - {k}: {v}")

discovery_rows = []
for feed_name, feed_url in feed_map.items():
    discovery_rows.append({
        "feed_name": feed_name,
        "feed_url": feed_url,
        "root_last_updated": gbfs_root.get("last_updated"),
        "root_ttl": gbfs_root.get("ttl")
    })

df_discovery = create_df_from_records(discovery_rows)
if df_discovery is not None:
    df_discovery = add_ingestion_columns(df_discovery, ingest_ts, ingest_epoch, GBFS_AUTO_DISCOVERY_URL)
    write_uc_table(df_discovery, TBL_DISCOVERY)
    print(f"Discovery saved: {TBL_DISCOVERY}")

# ------------------------------------------------------------
# 5) SYSTEM INFORMATION
# ------------------------------------------------------------
if "system_information" in feed_map:
    try:
        url = feed_map["system_information"]
        payload = fetch_json(url)
        write_raw_json(payload, "system_information", ingest_dt)

        record = payload.get("data", {}) or {}
        record["feed_last_updated"] = payload.get("last_updated")
        record["feed_ttl"] = payload.get("ttl")

        df = create_df_from_records([record])
        if df is not None:
            df = add_ingestion_columns(df, ingest_ts, ingest_epoch, url)
            write_uc_table(df, TBL_SYSTEM_INFO)
            print(f"system_information saved: {TBL_SYSTEM_INFO}")
    except Exception as e:
        print(f"Error processing system_information: {e}")
else:
    print("system_information not found in discovery.")

# ------------------------------------------------------------
# 6) STATION INFORMATION
# ------------------------------------------------------------
if "station_information" in feed_map:
    try:
        url = feed_map["station_information"]
        payload = fetch_json(url)
        write_raw_json(payload, "station_information", ingest_dt)

        stations = payload.get("data", {}).get("stations", []) or []
        print(f"station_information rows: {len(stations)}")

        for r in stations:
            r["feed_last_updated"] = payload.get("last_updated")
            r["feed_ttl"] = payload.get("ttl")

        df = create_df_from_records(stations)
        if df is not None:
            df = add_ingestion_columns(df, ingest_ts, ingest_epoch, url)
            write_uc_table(df, TBL_STATION_INFO)
            print(f"station_information saved: {TBL_STATION_INFO}")
    except Exception as e:
        print(f"Error processing station_information: {e}")
else:
    print("station_information not found in discovery.")

# ------------------------------------------------------------
# 7) STATION STATUS
# ------------------------------------------------------------
if "station_status" in feed_map:
    try:
        url = feed_map["station_status"]
        payload = fetch_json(url)
        write_raw_json(payload, "station_status", ingest_dt)

        stations = payload.get("data", {}).get("stations", []) or []
        print(f"station_status rows: {len(stations)}")

        for r in stations:
            r["feed_last_updated"] = payload.get("last_updated")
            r["feed_ttl"] = payload.get("ttl")

        df = create_df_from_records(stations)
        if df is not None:
            df = add_ingestion_columns(df, ingest_ts, ingest_epoch, url)
            write_uc_table(df, TBL_STATION_STATUS)
            print(f"station_status saved: {TBL_STATION_STATUS}")
    except Exception as e:
        print(f"Error processing station_status: {e}")
else:
    print("station_status not found in discovery.")

# ------------------------------------------------------------
# 8) FREE BIKE STATUS (OPTIONAL)
# ------------------------------------------------------------
if "free_bike_status" in feed_map:
    try:
        url = feed_map["free_bike_status"]
        payload = fetch_json(url)
        write_raw_json(payload, "free_bike_status", ingest_dt)

        bikes = payload.get("data", {}).get("bikes", []) or []
        print(f"free_bike_status rows: {len(bikes)}")

        for r in bikes:
            r["feed_last_updated"] = payload.get("last_updated")
            r["feed_ttl"] = payload.get("ttl")

        df = create_df_from_records(bikes)
        if df is not None:
            df = add_ingestion_columns(df, ingest_ts, ingest_epoch, url)
            write_uc_table(df, TBL_FREE_BIKE_STATUS)
            print(f"free_bike_status saved: {TBL_FREE_BIKE_STATUS}")
    except Exception as e:
        print(f"Error processing free_bike_status: {e}")
else:
    print("free_bike_status not published by this GBFS feed.")

# ------------------------------------------------------------
# 9) VEHICLE TYPES (OPTIONAL)
# ------------------------------------------------------------
if "vehicle_types" in feed_map:
    try:
        url = feed_map["vehicle_types"]
        payload = fetch_json(url)
        write_raw_json(payload, "vehicle_types", ingest_dt)

        vtypes = payload.get("data", {}).get("vehicle_types", []) or []
        print(f"vehicle_types rows: {len(vtypes)}")

        for r in vtypes:
            r["feed_last_updated"] = payload.get("last_updated")
            r["feed_ttl"] = payload.get("ttl")

        df = create_df_from_records(vtypes)
        if df is not None:
            df = add_ingestion_columns(df, ingest_ts, ingest_epoch, url)
            write_uc_table(df, TBL_VEHICLE_TYPES)
            print(f"vehicle_types saved: {TBL_VEHICLE_TYPES}")
    except Exception as e:
        print(f"Error processing vehicle_types: {e}")
else:
    print("vehicle_types not published by this GBFS feed.")

# ------------------------------------------------------------
# 10) LATEST VIEWS
# ------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {VW_STATION_STATUS_LATEST} AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY station_id
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_STATION_STATUS}
)
SELECT *
FROM ranked
WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {VW_STATION_INFO_LATEST} AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY station_id
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_STATION_INFO}
)
SELECT *
FROM ranked
WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {VW_SYSTEM_INFO_LATEST} AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_SYSTEM_INFO}
)
SELECT *
FROM ranked
WHERE rn = 1
""")

print("Views refreshed")

# ------------------------------------------------------------
# 11) OPTIONAL CHECKS
# ------------------------------------------------------------
try:
    print("station_information sample")
    display(spark.table(TBL_STATION_INFO).limit(10))
except Exception as e:
    print(f"Could not display station_information sample: {e}")

try:
    print("station_status sample")
except Exception as e:
    print(f"Could not display station_status sample: {e}")

print("========================================")
print("GBFS ingestion finished successfully")
print("========================================")

## Outputs

This notebook produces two types of outputs:

### 1. Raw JSON snapshots
Raw payloads are saved in DBFS under the GBFS raw ingestion path for traceability and reprocessing if needed.

Examples include:
- discovery feed
- station information
- station status
- vehicle types
- system information

---

### 2. Bronze Unity Catalog tables
The notebook writes normalized records into managed Delta tables, including:

- `workspace.default.bronze_gbfs_discovery`
- `workspace.default.bronze_gbfs_system_information`
- `workspace.default.bronze_gbfs_station_information`
- `workspace.default.bronze_gbfs_station_status`
- `workspace.default.bronze_gbfs_vehicle_types`

---

### 3. Latest operational views
The following views are refreshed to expose the newest system snapshot:

- `workspace.default.vw_gbfs_station_status_latest`
- `workspace.default.vw_gbfs_station_information_latest`
- `workspace.default.vw_gbfs_system_information_latest`

These views are essential for downstream feature engineering and serving jobs that require the current station state.

## Key Insights and Summary

### 1. The ingestion layer captures the real-time operational state of the system
This notebook provides the most recent view of station availability, system metadata, and infrastructure status, which is fundamental for both monitoring and forecasting.

---

### 2. Historical snapshots are preserved
Because the job appends each ingestion event and stores raw JSON files, the pipeline supports:
- historical reconstruction
- auditing
- debugging
- backfilling if required

---

### 3. The notebook separates raw ingestion from downstream analytics
This design is important because it keeps the bronze layer simple, reproducible, and independent from feature engineering or modeling logic.

---

### 4. Latest views simplify downstream consumption
Instead of forcing downstream jobs to identify the newest record manually, the notebook exposes ready-to-use views with the latest station and system state.

This reduces complexity in later pipeline stages.

---

### 5. Practical business value
This ingestion job ensures that the forecasting system always starts from the most up-to-date operational snapshot of the bike-sharing network.

Without this layer, downstream jobs would not be able to:
- detect the current bike/dock balance
- build hourly station snapshots
- generate short-term prediction inputs

In short, this notebook is the **real-time data entry point** of the GBFS branch of the project.